# Train, Validation, and Test Sets

## Overview

When building a machine learning model, we should not train and evaluate the model on the same data.

A proper machine learning workflow separates the available dataset into different parts:

- Training set
- Validation set
- Test set

Each set has a different purpose.

The main goal is to estimate how well the trained model will perform on unseen data.

---

## Learning Objectives

In this notebook, we will learn:

1. Why data is divided into different sets
2. Training set vs validation set vs test set
3. Common train-validation-test split strategies
4. How to use `train_test_split`
5. How to create train, validation, and test sets
6. Why stratification is important for classification
7. Random state and reproducibility
8. Data leakage
9. Why preprocessing must be fitted only on training data
10. Cross-validation
11. Hyperparameter tuning using validation data
12. Final evaluation using the test set
13. Common mistakes
14. A complete machine learning workflow

# 1. Why Do We Split Data?

Suppose we have a dataset containing 10,000 examples.

We train a machine learning model using these examples.

If we evaluate the model on the exact same examples that were used for training, the evaluation may be misleading.

The model may have learned the training data very well without learning a general pattern.

This is called **overfitting**.

Therefore, we need data that the model has not seen during training.

The basic idea is:

$$
\text{Training Data} \rightarrow \text{Learn Parameters}
$$

$$
\text{Validation Data} \rightarrow \text{Choose Model / Hyperparameters}
$$

$$
\text{Test Data} \rightarrow \text{Final Evaluation}
$$

# 2. Train, Validation, and Test Sets

## Training Set

The training set is used to learn the parameters of the machine learning model.

For example, in linear regression:

$$
\hat{y} = Xw + b
$$

The model learns:

- $w$
- $b$

using the training data.

---

## Validation Set

The validation set is used during model development.

It helps us make decisions such as:

- Which model should we use?
- Which hyperparameters should we choose?
- How much regularization should we use?
- Which model performs better?

The validation data should not be used to directly fit the model parameters.

---

## Test Set

The test set is used for the final evaluation.

It should remain untouched until we have finished:

- model selection
- hyperparameter tuning
- preprocessing decisions
- feature selection

The test set gives us an estimate of performance on unseen data.

---

## Important Principle

The test set should be treated as completely unseen data.

$$
\boxed{
\text{Never use the test set to make model decisions}
}
$$

# 3. Data Splitting Concept

A dataset can be divided like this:

$$
\text{Dataset}
\rightarrow
\begin{cases}
\text{Training Set} \\
\text{Validation Set} \\
\text{Test Set}
\end{cases}
$$

A common split is:

$$
70\% \rightarrow \text{Training}
$$

$$
15\% \rightarrow \text{Validation}
$$

$$
15\% \rightarrow \text{Test}
$$

Another common approach is:

$$
80\% \rightarrow \text{Training}
$$

$$
20\% \rightarrow \text{Test}
$$

with cross-validation performed on the training portion.

There is no single universally correct split ratio.

The appropriate strategy depends on:

- Dataset size
- Computational cost
- Model complexity
- Need for validation
- Availability of data

## 4. Import Libraries

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

# 5. Load a Dataset

We will use the Iris dataset for demonstration.

The dataset contains:

- 150 samples
- 4 numerical features
- 3 classes

The task is to predict the species of an iris flower from its measurements.

In [2]:
iris = load_iris()

X = iris.data
y = iris.target

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (150, 4)
y shape: (150,)


In [3]:
print("Feature names:")
print(iris.feature_names)

print("\nTarget names:")
print(iris.target_names)

Feature names:
['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)']

Target names:
['setosa' 'versicolor' 'virginica']


# 6. Train-Test Split

The simplest approach is to divide the dataset into:

- Training set
- Test set

Scikit-learn provides:

`train_test_split()`

The `test_size` parameter controls the fraction of data assigned to the test set.

For example:

$$
test\_size = 0.2
$$

means:

$$
80\% \rightarrow \text{Training}
$$

$$
20\% \rightarrow \text{Testing}
$$

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training samples:", len(X_train))
print("Test samples:", len(X_test))

Training samples: 120
Test samples: 30


# 7. Random State

`train_test_split()` performs a random split.

Therefore, different executions can produce different splits.

We can control this using:

`random_state`

For example:


In [5]:
random_state=42

The specific number 42 has no special mathematical meaning.

It is simply a commonly used seed.

The important idea is reproducibility.

If we use the same random state:

$$ \text{Same Dataset} + \text{Same Random State} \rightarrow \text{Same Split} $$

This makes experiments reproducible.


---

# 8. Creating Train, Validation, and Test Sets

To create three sets, we can perform the split in two steps.

Suppose we want:

$$
70\% \rightarrow \text{Training}
$$

$$
15\% \rightarrow \text{Validation}
$$

$$
15\% \rightarrow \text{Test}
$$

First, we separate the test set:

$$
85\% \rightarrow \text{Temporary Training Data}
$$

$$
15\% \rightarrow \text{Test}
$$

Then we split the temporary training data:

$$
\frac{15}{85} \approx 17.65\%
$$

of the temporary data becomes validation data.

The final proportions are approximately:

- 70% training
- 15% validation
- 15% test

In [6]:
X_train_temp, X_test, y_train_temp, y_test = train_test_split(
    X,
    y,
    test_size=0.15,
    random_state=42
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_temp,
    y_train_temp,
    test_size=0.1765,
    random_state=42
)

print("Training samples:", len(X_train))
print("Validation samples:", len(X_val))
print("Test samples:", len(X_test))

Training samples: 104
Validation samples: 23
Test samples: 23


In [7]:
total = len(X)

print("Training:", len(X_train) / total)
print("Validation:", len(X_val) / total)
print("Test:", len(X_test) / total)

Training: 0.6933333333333334
Validation: 0.15333333333333332
Test: 0.15333333333333332


# 9. Why Do We Need Validation Data?

Suppose we want to compare three models:

- Logistic Regression
- Decision Tree
- K-Nearest Neighbors

We train all three models on the training set.

Then we evaluate them on the validation set.

For example:

| Model | Validation Accuracy |
|---|---:|
| Logistic Regression | 0.96 |
| Decision Tree | 0.93 |
| KNN | 0.97 |

We would select KNN because it performs best on the validation set.

The test set remains untouched.

After selecting the final model, we evaluate it once on the test set.

This gives a more honest estimate of generalization performance.

# 10. Stratification

In classification problems, different classes may have different frequencies.

For example:

$$
\text{Class A} = 70\%
$$

$$
\text{Class B} = 20\%
$$

$$
\text{Class C} = 10\%
$$

A random split could accidentally produce different class proportions in the training and test sets.

To reduce this problem, we can use:

`stratify=y`

Stratification attempts to preserve the class distribution across the splits.

For classification problems, stratification is often a good default when the dataset is not extremely small or otherwise constrained.

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training class distribution:")
print(np.bincount(y_train))

print("\nTest class distribution:")
print(np.bincount(y_test))

Training class distribution:
[40 40 40]

Test class distribution:
[10 10 10]


## 11. Comparing Class Proportions

In [9]:
print("Original distribution:")
print(np.bincount(y) / len(y))

print("\nTraining distribution:")
print(np.bincount(y_train) / len(y_train))

print("\nTest distribution:")
print(np.bincount(y_test) / len(y_test))

Original distribution:
[0.33333333 0.33333333 0.33333333]

Training distribution:
[0.33333333 0.33333333 0.33333333]

Test distribution:
[0.33333333 0.33333333 0.33333333]


# 12. Data Leakage

Data leakage occurs when information from outside the training data improperly influences the model during training.

This is one of the most important problems in machine learning.

Leakage can happen when we accidentally use:

- Test data during preprocessing
- Future information
- Target information
- Statistics calculated from the entire dataset
- Features that would not actually be available at prediction time

---

## Example

Suppose we standardize the entire dataset before splitting:

```python
scaler.fit_transform(X)

The scaler calculates:

Mean
Standard deviation

using all observations.

That means information from the future validation/test data has influenced the transformation.

This is a form of data leakage.

Instead, we should:

$$ \text{Fit preprocessing on training data only} $$

and then:

$$ \text{Transform validation/test data using the fitted preprocessing} $$


---


# 13. Correct Preprocessing Workflow

Suppose we use standardization:

$$
z = \frac{x-\mu}{\sigma}
$$

The values $\mu$ and $\sigma$ must be calculated using the training set only.

Correct workflow:

$$
X_{train}
\rightarrow
\text{fit scaler}
$$

Then:

$$
X_{train}
\rightarrow
\text{transform}
$$

$$
X_{validation}
\rightarrow
\text{transform}
$$

$$
X_{test}
\rightarrow
\text{transform}
$$

We never calculate new scaling statistics using validation or test data.

In [11]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_test_scaled = scaler.transform(X_test)

print("Training mean:")
print(X_train_scaled.mean(axis=0))

print("\nTraining standard deviation:")
print(X_train_scaled.std(axis=0))

Training mean:
[-1.20829273e-15 -2.03679666e-15  4.99600361e-16  1.67458640e-15]

Training standard deviation:
[1. 1. 1. 1.]


# 14. `fit_transform()` vs `transform()`

For training data:

```python
scaler.fit_transform(X_train)

This does two operations:

Learn the scaling parameters
Transform the training data

For validation or test data:

In [12]:
scaler.transform(X_test)

array([[-1.72156775, -0.10821272, -1.40250384, -1.32327558],
       [ 0.30848902, -0.10821272,  0.64163131,  0.78343181],
       [-1.12449223, -1.45154306, -0.2668732 , -0.26992188],
       [-1.00507713, -1.67543145, -0.2668732 , -0.26992188],
       [-1.72156775,  0.33956406, -1.40250384, -1.32327558],
       [ 0.54731923,  0.56345245,  0.52806825,  0.52009339],
       [-1.48273754,  1.23511762, -1.57284844, -1.32327558],
       [-0.52741671,  0.78734084, -1.17537771, -1.32327558],
       [ 0.78614944, -0.10821272,  0.81197591,  1.04677024],
       [-0.52741671, -0.10821272,  0.41450518,  0.38842418],
       [ 1.74147027, -0.33210111,  1.43657276,  0.78343181],
       [ 1.26380985,  0.11567567,  0.75519438,  1.44177787],
       [ 0.78614944, -0.10821272,  1.1526651 ,  1.31010866],
       [ 0.66673433,  0.33956406,  0.41450518,  0.38842418],
       [-1.00507713,  0.78734084, -1.28894078, -1.32327558],
       [-1.00507713,  0.56345245, -1.34572231, -1.32327558],
       [-0.04975629,  2.

This only transforms the data using parameters already learned from the training set.

Therefore:

$$ \boxed{ \text{Training: fit + transform} } $$
Validation/Test: transform only
	​



---
# 15. Using a Pipeline

A `Pipeline` can automatically keep preprocessing and model training together.

This is especially useful because it reduces the risk of data leakage.

The pipeline performs:

$$
\text{Raw Data}
\rightarrow
\text{Preprocessing}
\rightarrow
\text{Model}
$$

When cross-validation is used, the preprocessing is fitted separately inside each training fold.

This is the preferred approach for many practical machine learning workflows.

In [13]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=1000))
])

model.fit(X_train, y_train)

train_accuracy = model.score(X_train, y_train)
test_accuracy = model.score(X_test, y_test)

print("Training accuracy:", train_accuracy)
print("Test accuracy:", test_accuracy)

Training accuracy: 0.9583333333333334
Test accuracy: 0.9333333333333333


# 16. Cross-Validation

A single validation split can give an unstable estimate of model performance.

Cross-validation provides a more robust evaluation method.

In $K$-fold cross-validation:

1. Divide the training data into $K$ folds.
2. Train on $K-1$ folds.
3. Validate on the remaining fold.
4. Repeat until every fold has been used for validation.
5. Calculate the average score.

For $K=5$:

$$
\text{Fold 1} \rightarrow \text{Validation}
$$

$$
\text{Fold 2} \rightarrow \text{Validation}
$$

$$
\text{Fold 3} \rightarrow \text{Validation}
$$

$$
\text{Fold 4} \rightarrow \text{Validation}
$$

$$
\text{Fold 5} \rightarrow \text{Validation}
$$

The final score is usually the mean of the five validation scores.

## 17. K-Fold Cross-Validation

In [14]:
from sklearn.model_selection import cross_val_score

model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=1000))
])

cv_scores = cross_val_score(
    model,
    X_train,
    y_train,
    cv=5,
    scoring="accuracy"
)

print("Cross-validation scores:")
print(cv_scores)

print("\nMean CV accuracy:", cv_scores.mean())
print("Standard deviation:", cv_scores.std())

Cross-validation scores:
[0.91666667 0.95833333 0.95833333 0.95833333 1.        ]

Mean CV accuracy: 0.9583333333333334
Standard deviation: 0.026352313834736508


# 18. Why Use Cross-Validation?

Cross-validation gives us multiple validation results instead of relying on a single split.

For example:

$$
[0.95,\ 0.97,\ 0.94,\ 0.96,\ 0.98]
$$

Mean:

$$
\text{Mean CV Score}
=
\frac{1}{K}
\sum_{i=1}^{K} Score_i
$$

The standard deviation tells us how much the performance varies between folds.

A small standard deviation generally indicates more consistent performance across the folds.

---

## Advantages

- Better use of limited data
- More stable performance estimate
- Useful for model comparison
- Useful for hyperparameter tuning

## Disadvantages

- More computationally expensive
- Can be unnecessary for extremely large datasets
- Must be implemented carefully to avoid leakage

# 19. Hyperparameter Tuning

Model parameters are learned from training data.

Hyperparameters are chosen by us before or during model development.

Examples:

- Learning rate
- Regularization strength
- Number of neighbors
- Tree depth
- Number of trees

Suppose we want to choose the best value of $C$ for logistic regression.

We can evaluate several values using cross-validation.

The general workflow is:

$$
\text{Training Data}
\rightarrow
\text{Cross-Validation}
\rightarrow
\text{Select Hyperparameters}
$$

The test set should not be used to choose the hyperparameters.

## 20. Grid Search

In [15]:
from sklearn.model_selection import GridSearchCV

pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=1000))
])

param_grid = {
    "classifier__C": [0.01, 0.1, 1, 10, 100]
}

grid_search = GridSearchCV(
    pipeline,
    param_grid=param_grid,
    cv=5,
    scoring="accuracy"
)

grid_search.fit(X_train, y_train)

print("Best parameters:")
print(grid_search.best_params_)

print("\nBest CV score:")
print(grid_search.best_score_)

Best parameters:
{'classifier__C': 10}

Best CV score:
0.9666666666666668


# 21. Final Test Evaluation

After selecting the model and hyperparameters using the training data and cross-validation, we evaluate the final model on the test set.

The test set has not been used for:

- Model training
- Hyperparameter selection
- Model selection
- Preprocessing fitting

Therefore, it provides an approximately unbiased estimate of how the final model performs on unseen data.

The workflow is:

$$
\text{Training Data}
\rightarrow
\text{Model Selection}
\rightarrow
\text{Hyperparameter Tuning}
$$

Then:

$$
\text{Final Model}
\rightarrow
\text{Test Set}
\rightarrow
\text{Final Performance}
$$

In [16]:
from sklearn.metrics import accuracy_score, classification_report

best_model = grid_search.best_estimator_

y_test_pred = best_model.predict(X_test)

test_accuracy = accuracy_score(y_test, y_test_pred)

print("Final Test Accuracy:", test_accuracy)

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_test_pred,
    target_names=iris.target_names
))

Final Test Accuracy: 1.0

Classification Report:
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       1.00      1.00      1.00        10
   virginica       1.00      1.00      1.00        10

    accuracy                           1.00        30
   macro avg       1.00      1.00      1.00        30
weighted avg       1.00      1.00      1.00        30



# 22. Comparing Training and Test Performance

It is useful to compare training and test performance.

A very high training score with a significantly lower validation/test score may indicate overfitting.

For example:

$$
\text{Training Accuracy} = 99\%
$$

$$
\text{Test Accuracy} = 82\%
$$

This large difference suggests that the model may not generalize well.

A smaller gap is generally preferable, although the acceptable gap depends on the problem and model.

In [17]:
train_accuracy = best_model.score(X_train, y_train)
test_accuracy = best_model.score(X_test, y_test)

print("Training Accuracy:", train_accuracy)
print("Test Accuracy:", test_accuracy)

Training Accuracy: 0.975
Test Accuracy: 1.0


# 23. Underfitting vs Overfitting

## Underfitting

The model is too simple to capture the underlying pattern.

Typical behavior:

$$
\text{Training Performance: Low}
$$

$$
\text{Test Performance: Low}
$$

---

## Good Generalization

The model learns useful patterns without memorizing the training data.

$$
\text{Training Performance: Good}
$$

$$
\text{Test Performance: Good}
$$

---

## Overfitting

The model learns the training data too closely.

Typical behavior:

$$
\text{Training Performance: Very High}
$$

$$
\text{Test Performance: Significantly Lower}
$$

The goal of machine learning is not simply to maximize training performance.

The goal is:

$$
\boxed{
\text{Good Generalization to Unseen Data}
}
$$

# 24. Common Mistakes

## Mistake 1: Training and testing on the same data

```text

model.fit(X, y)
model.predict(X)
This does not provide a reliable estimate of generalization.

Mistake 2: Fitting preprocessing on the entire dataset

Incorrect:

scaler.fit(X)

before splitting the data.

Correct:

scaler.fit(X_train)
Mistake 3: Using the test set for hyperparameter tuning

If we repeatedly compare models using the test set, the test set is no longer truly unseen.

Mistake 4: Ignoring class imbalance

For classification problems, random splitting without considering class proportions can produce poor splits.

Using stratification can help:

stratify=y
Mistake 5: Forgetting reproducibility

Without a fixed random state, different runs may produce different splits.

Mistake 6: Performing preprocessing outside cross-validation

Preprocessing should be performed within each training fold.

Using a Pipeline is a reliable way to achieve this.


---
# 25. Complete Machine Learning Workflow

A practical supervised learning workflow can be summarized as follows:

### Step 1 — Collect Data

Obtain the dataset.

### Step 2 — Split the Data

Separate a final test set.

$$
\text{Dataset}
\rightarrow
\text{Training Data}
+
\text{Test Data}
$$

### Step 3 — Preprocess Training Data

Fit preprocessing only on the training data.

### Step 4 — Model Development

Train candidate models using the training data.

### Step 5 — Cross-Validation

Use cross-validation to compare models and tune hyperparameters.

### Step 6 — Select the Final Model

Choose the model and hyperparameters based on training/CV results.

### Step 7 — Final Test Evaluation

Evaluate the selected model once on the untouched test set.

### Step 8 — Deploy

After satisfactory evaluation, train/deploy according to the chosen production workflow.

---

## Overall Flow

$$
\boxed{
\text{Data}
\rightarrow
\text{Train/Test Split}
\rightarrow
\text{CV + Tuning}
\rightarrow
\text{Final Model}
\rightarrow
\text{Test Evaluation}
\rightarrow
\text{Deployment}
}
$$

# 26. Important Concepts to Remember

| Concept | Purpose |
|---|---|
| Training Set | Learn model parameters |
| Validation Set | Select models and hyperparameters |
| Test Set | Final unbiased evaluation |
| `train_test_split()` | Split data |
| `random_state` | Reproducibility |
| `stratify` | Preserve class proportions |
| Cross-validation | More reliable model comparison |
| Pipeline | Combine preprocessing and model safely |
| Data leakage | Prevent information from improperly crossing the split |
| Hyperparameter tuning | Find good model settings |
| Generalization | Performance on unseen data |

---

## Most Important Rule

The test set should remain untouched until the final evaluation.

$$
\boxed{
\text{Do not make model decisions using the test set}
}
$$

# 27. Summary

In this notebook, we learned why machine learning datasets are divided into training, validation, and test sets.

The key ideas are:

1. The **training set** is used to learn model parameters.
2. The **validation set** is used for model selection and hyperparameter tuning.
3. The **test set** is reserved for final evaluation.
4. `random_state` makes random splits reproducible.
5. `stratify` helps preserve class distributions in classification.
6. Data leakage can produce overly optimistic results.
7. Preprocessing should be fitted only on training data.
8. Pipelines help prevent preprocessing leakage.
9. Cross-validation provides multiple validation estimates.
10. Hyperparameters should be selected without using the test set.
11. The final test evaluation should happen after model selection.
12. The ultimate goal is **generalization**, not memorization.

---

## Core Principle

$$
\boxed{
\text{Train on training data}
}
$$

$$
\boxed{
\text{Choose using validation / cross-validation}
}
$$

$$
\boxed{
\text{Evaluate once on the test data}
}
$$

This separation is fundamental to building reliable machine learning models.